# Sparse autoencoder для интерпретации MLP-активаций

В ноутбуке разбирается sparse autoencoder (SAE) для однослойной модели `gelu-1l` из TransformerLens. Мы обучим небольшой SAE, сравним его с готовым checkpoint и посмотрим, какие токены и logits связаны с отдельными признаками.

Зависимости должны быть установлены в окружении заранее. Ноутбук не меняет окружение во время запуска.


In [25]:
from functools import partial

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
import tqdm.notebook as tqdm
from datasets import load_dataset
from transformer_lens import HookedTransformer, utils


DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
DTYPES = {
    "fp32": torch.float32,
    "fp16": torch.float16,
    "bf16": torch.bfloat16,
}
PLOTLY_TEMPLATE = "plotly_white"

print(f"Устройство: {DEVICE}")


Устройство: cuda:1


## Устройство sparse autoencoder


SAE получает MLP-активацию размерности `d_mlp` и раскладывает её по переопределённому словарю признаков. Размер словаря равен `d_mlp * dict_mult`, поэтому признаков больше, чем исходных координат.

Encoder сначала вычитает обучаемое смещение `b_dec`, затем вычисляет неотрицательные активации признаков. Decoder собирает из них приближение исходного вектора. Функция потерь состоит из ошибки реконструкции и L1-штрафа: первый член сохраняет информацию, второй делает представление разреженным.


In [2]:
cfg = {
    "seed": 49,
    "batch_size": 4096,
    "buffer_mult": 384,
    "lr": 1e-4,
    "num_tokens": int(2e9),
    "l1_coeff": 3e-4,
    "beta1": 0.9,
    "beta2": 0.99,
    "dict_mult": 8,
    "seq_len": 128,
    "d_mlp": 2048,
    "enc_dtype": "fp32",
    "remove_rare_dir": False,
}

cfg["model_batch_size"] = 64
cfg["buffer_size"] = cfg["batch_size"] * cfg["buffer_mult"]
cfg["buffer_batches"] = cfg["buffer_size"] // cfg["seq_len"]


In [3]:
class AutoEncoder(nn.Module):
    def __init__(self, config, device=DEVICE):
        super().__init__()

        d_mlp = config["d_mlp"]
        d_hidden = d_mlp * config["dict_mult"]
        dtype = DTYPES[config["enc_dtype"]]

        torch.manual_seed(config["seed"])
        self.W_enc = nn.Parameter(
            nn.init.kaiming_uniform_(torch.empty(d_mlp, d_hidden, dtype=dtype))
        )
        self.W_dec = nn.Parameter(
            nn.init.kaiming_uniform_(torch.empty(d_hidden, d_mlp, dtype=dtype))
        )
        self.b_enc = nn.Parameter(torch.zeros(d_hidden, dtype=dtype))
        self.b_dec = nn.Parameter(torch.zeros(d_mlp, dtype=dtype))

        self.d_hidden = d_hidden
        self.l1_coeff = config["l1_coeff"]
        self.to(device)
        self.normalize_decoder_weights()

    def encode(self, x):
        # Центрирование относительно b_dec согласует вход encoder с bias decoder.
        centered_x = x - self.b_dec
        return F.relu(centered_x @ self.W_enc + self.b_enc)

    def decode(self, feature_acts):
        return feature_acts @ self.W_dec + self.b_dec

    def forward(self, x):
        feature_acts = self.encode(x)
        reconstructed_x = self.decode(feature_acts)

        l2_loss = (reconstructed_x.float() - x.float()).pow(2).sum(-1).mean()
        l1_loss = self.l1_coeff * feature_acts.float().abs().sum()
        loss = l2_loss + l1_loss
        return loss, reconstructed_x, feature_acts, l2_loss, l1_loss

    @torch.no_grad()
    def normalize_decoder_weights(self):
        norms = self.W_dec.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        self.W_dec.div_(norms)

    @torch.no_grad()
    def remove_parallel_component_of_grads(self):
        if self.W_dec.grad is None:
            return

        # У decoder-направлений фиксирована единичная норма. Удаляем часть
        # градиента вдоль самого направления, чтобы шаг optimizer её не менял.
        decoder_directions = self.W_dec / self.W_dec.norm(
            dim=-1, keepdim=True
        ).clamp_min(1e-8)
        parallel_grad = (
            (self.W_dec.grad * decoder_directions).sum(-1, keepdim=True)
            * decoder_directions
        )
        self.W_dec.grad.sub_(parallel_grad)

    @classmethod
    def load_from_hf(cls, version, device=DEVICE):
        checkpoints = {"run1": 25, "run2": 47}
        checkpoint = checkpoints.get(version, version)

        config = utils.download_file_from_hf(
            "NeelNanda/sparse_autoencoder",
            f"{checkpoint}_cfg.json",
        )
        state_dict = utils.download_file_from_hf(
            "NeelNanda/sparse_autoencoder",
            f"{checkpoint}.pt",
            force_is_torch=True,
        )

        autoencoder = cls(config=config, device=device)
        autoencoder.load_state_dict(state_dict)
        print(f"Загружен SAE checkpoint {checkpoint}: {config}")
        return autoencoder


## Метрики качества


### Сохранение поведения модели


Хук ставится на `blocks.0.mlp.hook_post`, где тензор имеет форму `[batch, seq_len, d_mlp]`. В одном запуске модель получает исходные MLP-активации, во втором — реконструкцию SAE, в третьем — нули вместо всего MLP-выхода.

Итоговый reconstruction score показывает, какую долю ухудшения от zero ablation восстанавливает SAE. Значение около 1 означает, что реконструкция почти сохраняет поведение исходной модели; значение около 0 означает, что она не лучше зануления слоя.


In [4]:
def replacement_hook(mlp_post, hook, local_encoder):
    return local_encoder(mlp_post)[1]


def zero_ablate_hook(mlp_post, hook):
    return torch.zeros_like(mlp_post)


@torch.no_grad()
def get_recons_loss(num_batches=5, local_encoder=None):
    local_encoder = local_encoder or encoder
    hook_name = utils.get_act_name("post", 0)
    batch_losses = []

    for _ in range(num_batches):
        indices = torch.randperm(len(all_tokens))[: cfg["model_batch_size"]]
        tokens = all_tokens[indices]

        model_loss = model(tokens, return_type="loss")
        reconstruction_loss = model.run_with_hooks(
            tokens,
            return_type="loss",
            fwd_hooks=[
                (
                    hook_name,
                    partial(replacement_hook, local_encoder=local_encoder),
                )
            ],
        )
        zero_ablation_loss = model.run_with_hooks(
            tokens,
            return_type="loss",
            fwd_hooks=[(hook_name, zero_ablate_hook)],
        )
        batch_losses.append(
            torch.stack([model_loss, reconstruction_loss, zero_ablation_loss])
        )

    model_loss, reconstruction_loss, zero_ablation_loss = (
        torch.stack(batch_losses).mean(0).tolist()
    )
    score = (zero_ablation_loss - reconstruction_loss) / (
        zero_ablation_loss - model_loss
    )

    print(
        f"loss: {model_loss:.4f}, "
        f"reconstruction: {reconstruction_loss:.4f}, "
        f"zero ablation: {zero_ablation_loss:.4f}"
    )
    print(f"Reconstruction score: {score:.2%}")
    return score, model_loss, reconstruction_loss, zero_ablation_loss


### Частота активации признаков


Для каждого признака считаем долю токенов, на которых его активация больше нуля. Это простая оценка feature density: содержательные признаки обычно срабатывают лишь в определённых контекстах, а постоянно активные признаки плохо соответствуют идее разреженного представления.

Нулевая частота на конечной выборке ещё не доказывает, что признак полностью мёртв, но помогает быстро найти кандидатов для отдельного анализа.


In [5]:
@torch.no_grad()
def get_feature_frequencies(num_batches=25, local_encoder=None):
    local_encoder = local_encoder or encoder
    hook_name = utils.get_act_name("post", 0)
    active_counts = torch.zeros(
        local_encoder.d_hidden,
        dtype=torch.float32,
        device=local_encoder.W_dec.device,
    )
    token_count = 0

    for _ in tqdm.trange(num_batches):
        indices = torch.randperm(len(all_tokens))[: cfg["model_batch_size"]]
        tokens = all_tokens[indices]
        _, cache = model.run_with_cache(
            tokens,
            stop_at_layer=1,
            names_filter=hook_name,
        )
        mlp_acts = cache[hook_name].reshape(-1, model.cfg.d_mlp)
        feature_acts = local_encoder.encode(mlp_acts)

        active_counts += (feature_acts > 0).sum(0)
        token_count += feature_acts.shape[0]

    frequencies = active_counts / token_count
    dead_fraction = (frequencies == 0).float().mean().item()
    print(f"Доля признаков без активаций в выборке: {dead_fraction:.2%}")
    return frequencies


## Утилиты для токенов и визуализации

В таблицах пробел обозначается символом `·`, перенос строки — `↩`, табуляция — `→`. Так границы токенов остаются видимыми и короткие фрагменты легче читать.


### Преобразование токенов и logits


In [6]:
SPACE = "·"
NEWLINE = "↩"
TAB = "→"


def process_token(token):
    if isinstance(token, torch.Tensor):
        token = token.item()
    if isinstance(token, np.integer):
        token = token.item()
    if isinstance(token, int):
        token = model.to_string(token)

    return token.replace(" ", SPACE).replace("\n", NEWLINE + "\n").replace("\t", TAB)


def process_tokens(tokens):
    if isinstance(tokens, str):
        tokens = model.to_str_tokens(tokens)
    elif isinstance(tokens, torch.Tensor) and tokens.ndim > 1:
        tokens = tokens.squeeze(0)
    return [process_token(token) for token in tokens]


def create_vocab_df(logit_vector, make_probs=False, full_vocab=None):
    if full_vocab is None:
        token_ids = torch.arange(model.cfg.d_vocab)
        full_vocab = process_tokens(model.to_str_tokens(token_ids))

    vocab_df = pd.DataFrame(
        {
            "token": full_vocab,
            "logit": utils.to_numpy(logit_vector),
        }
    )
    if make_probs:
        vocab_df["log_prob"] = utils.to_numpy(logit_vector.log_softmax(dim=-1))
        vocab_df["prob"] = utils.to_numpy(logit_vector.softmax(dim=-1))
    return vocab_df.sort_values("logit", ascending=False)


### Контексты и активации выбранного признака

`make_token_df` разворачивает батч токенов в таблицу и добавляет короткий контекст вокруг каждой позиции. Две следующие функции работают с произвольным текстом: первая возвращает численную таблицу, вторая строит Plotly-график по тем же данным.


In [7]:
def flatten(nested_list):
    return [item for group in nested_list for item in group]


def make_token_df(tokens, prefix_length=5, suffix_length=1):
    string_tokens = [process_tokens(model.to_str_tokens(row)) for row in tokens]
    rows = []

    for batch_index, sequence in enumerate(string_tokens):
        for position, current_token in enumerate(sequence):
            prefix = "".join(sequence[max(0, position - prefix_length) : position])
            suffix = "".join(sequence[position + 1 : position + 1 + suffix_length])
            rows.append(
                {
                    "str_tokens": current_token,
                    "unique_token": f"{current_token}/{position}",
                    "context": f"{prefix}|{current_token}|{suffix}",
                    "batch": batch_index,
                    "pos": position,
                    "label": f"{batch_index}/{position}",
                }
            )

    return pd.DataFrame(rows)


@torch.no_grad()
def feature_activation_table(text, feature_index, local_encoder=None):
    local_encoder = local_encoder or encoder
    hook_name = utils.get_act_name("post", 0)
    tokens = model.to_tokens(text)
    _, cache = model.run_with_cache(
        tokens,
        stop_at_layer=1,
        names_filter=hook_name,
    )

    mlp_acts = cache[hook_name][0]
    feature_acts = local_encoder.encode(mlp_acts)[:, feature_index]
    string_tokens = process_tokens(tokens[0])

    return pd.DataFrame(
        {
            "position": range(len(string_tokens)),
            "token": string_tokens,
            "activation": utils.to_numpy(feature_acts),
        }
    )


def plot_feature_activations(text, feature_index, local_encoder=None) -> go.Figure:
    activation_df = feature_activation_table(text, feature_index, local_encoder)
    figure = px.bar(
        activation_df,
        x="position",
        y="activation",
        hover_data={"token": True, "position": True, "activation": ":.4f"},
        title=f"Активации SAE-признака {feature_index} по токенам",
        labels={"position": "Позиция токена", "activation": "Активация"},
        template=PLOTLY_TEMPLATE,
    )
    figure.update_traces(marker_color="#3b82f6")
    figure.update_layout(hovermode="x unified")
    return figure


## Загрузка модели

`gelu-1l` удобна для учебного анализа: у неё один transformer block, поэтому влияние MLP-признака на итоговые logits можно проследить напрямую через decoder модели.


In [8]:
model = HookedTransformer.from_pretrained("gelu-1l")
model = model.to(DTYPES[cfg["enc_dtype"]]).to(DEVICE)

print(
    f"Слоёв: {model.cfg.n_layers}; "
    f"d_model: {model.cfg.d_model}; "
    f"d_mlp: {model.cfg.d_mlp}; "
    f"словарь: {model.cfg.d_vocab}"
)


Loaded pretrained model gelu-1l into HookedTransformer
Changing model dtype to torch.float32
Moving model to device:  cuda
Слоёв: 1; d_model: 512; d_mlp: 2048; словарь: 48262


## Загрузка данных

Используется небольшой срез C4 с кодом. TransformerLens токенизирует тексты, объединяет их в последовательности фиксированной длины и добавляет BOS-токен согласно настройкам tokenizer модели.


In [9]:
data = load_dataset("NeelNanda/c4-code-20k", split="train")
tokenized_data = utils.tokenize_and_concatenate(
    data,
    model.tokenizer,
    max_length=cfg["seq_len"],
)
all_tokens = tokenized_data.shuffle(seed=42)["tokens"]

print(f"Последовательностей в датасете: {len(all_tokens):,}")


Последовательностей в датасете: 215,399


## Обучение локального SAE

На каждом шаге берём несколько последовательностей, сохраняем `blocks.0.mlp.hook_post` и превращаем формы `[batch, seq_len, d_mlp]` в обычный батч MLP-векторов. SAE учится реконструировать эти векторы с L1-штрафом на активации признаков.

Локальный запуск намеренно короче полноценного обучения reference SAE. Он нужен для воспроизводимого примера всего pipeline, а не для достижения качества опубликованного checkpoint.


In [10]:
local_sae_cfg = cfg.copy()
local_sae_cfg.update(
    {
        "seed": 123,
        "dict_mult": 8,
        "batch_size": 2048,
        "lr": 1e-4,
        "l1_coeff": 3e-4,
        "enc_dtype": "fp32",
    }
)

TRAIN_LOCAL_SAE = True
LOCAL_SAE_STEPS = 1000
LOCAL_SAE_LOG_EVERY = 25
LOCAL_SAE_PATH = "local_sae_gelu_1l.pt"


In [11]:
@torch.no_grad()
def sample_mlp_post_acts(num_activation_vectors, token_source=None):
    token_source = all_tokens if token_source is None else token_source
    sequences_needed = (num_activation_vectors + cfg["seq_len"] - 1) // cfg["seq_len"]
    indices = torch.randperm(len(token_source))[:sequences_needed]
    tokens = token_source[indices]
    hook_name = utils.get_act_name("post", 0)

    _, cache = model.run_with_cache(
        tokens,
        stop_at_layer=1,
        names_filter=hook_name,
    )
    mlp_acts = cache[hook_name].reshape(-1, model.cfg.d_mlp)
    return mlp_acts[:num_activation_vectors].detach()


def train_local_sae(local_config, num_steps=300, log_every=25):
    local_encoder = AutoEncoder(local_config)
    optimizer = torch.optim.Adam(
        local_encoder.parameters(),
        lr=local_config["lr"],
        betas=(local_config["beta1"], local_config["beta2"]),
    )
    history = []
    progress = tqdm.trange(num_steps)

    for step in progress:
        mlp_acts = sample_mlp_post_acts(local_config["batch_size"])
        loss, reconstruction, feature_acts, l2_loss, l1_loss = local_encoder(mlp_acts)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        local_encoder.remove_parallel_component_of_grads()
        optimizer.step()
        local_encoder.normalize_decoder_weights()

        should_log = step == 0 or (step + 1) % log_every == 0 or step + 1 == num_steps
        if not should_log:
            continue

        with torch.no_grad():
            mse = (reconstruction.float() - mlp_acts.float()).pow(2).mean().item()
            mean_l0 = (feature_acts > 0).float().sum(dim=-1).mean().item()
            density = (feature_acts > 0).float().mean().item()

        history.append(
            {
                "step": step + 1,
                "loss": loss.item(),
                "l2_loss": l2_loss.item(),
                "l1_loss": l1_loss.item(),
                "mse_per_activation": mse,
                "mean_l0": mean_l0,
                "density": density,
            }
        )
        progress.set_postfix(
            loss=f"{loss.item():.2f}",
            mse=f"{mse:.4f}",
            l0=f"{mean_l0:.1f}",
        )

    return local_encoder, pd.DataFrame(history)


if TRAIN_LOCAL_SAE:
    trained_encoder, local_sae_history = train_local_sae(
        local_sae_cfg,
        num_steps=LOCAL_SAE_STEPS,
        log_every=LOCAL_SAE_LOG_EVERY,
    )
    torch.save(
        {"cfg": local_sae_cfg, "state_dict": trained_encoder.state_dict()},
        LOCAL_SAE_PATH,
    )
else:
    trained_encoder = None
    local_sae_history = pd.DataFrame()

local_sae_history.tail()


  0%|          | 0/1000 [00:00<?, ?it/s]

,step,loss,l2_loss,l1_loss,mse_per_activation,mean_l0,density
36,900,97.984703,74.117126,23.867579,0.036190,180.675293,0.011028
37,925,99.313278,75.538414,23.774860,0.036884,180.850586,0.011038
38,950,96.416351,72.687286,23.729063,0.035492,178.737793,0.010909
39,975,95.869431,72.262146,23.607286,0.035284,181.871094,0.011101
40,1000,92.347557,68.632957,23.714600,0.033512,183.847168,0.011221


Кривые получаются шумными, потому что каждый шаг использует свежую выборку активаций. Для быстрой проверки достаточно следить, что `mse_per_activation` снижается, а `mean_l0` остаётся заметно меньше полного размера словаря.

Для серьёзного сравнения нужны более длинное обучение, несколько seed и отдельная validation-выборка.


In [12]:
if not local_sae_history.empty:
    training_figure = px.line(
        local_sae_history,
        x="step",
        y=["mse_per_activation", "density"],
        title="Диагностика обучения локального SAE",
        labels={"step": "Шаг", "value": "Значение", "variable": "Метрика"},
        template=PLOTLY_TEMPLATE,
    )
    training_figure.show()


# Анализ SAE

## Загрузка reference checkpoint

В репозитории `NeelNanda/sparse_autoencoder` опубликованы два основных запуска с разными seed. `run1` соответствует checkpoint 25, `run2` — checkpoint 47. Дальше используем первый запуск как reference.


Здесь новый autoencoder не обучается: конфигурация и веса скачиваются с Hugging Face. Формат модели совпадает с локальным SAE, поэтому к обоим можно применять одни и те же метрики.


In [13]:
auto_encoder_run = "run1"
encoder = AutoEncoder.load_from_hf(auto_encoder_run)


Загружен SAE checkpoint 25: {'seed': 52, 'batch_size': 4096, 'buffer_mult': 384, 'lr': 0.0001, 'num_tokens': 2000000000, 'l1_coeff': 0.0003, 'beta1': 0.9, 'beta2': 0.99, 'dict_mult': 8, 'seq_len': 128, 'd_mlp': 2048, 'enc_dtype': 'fp32', 'model_batch_size': 512, 'buffer_size': 1572864, 'buffer_batches': 12288}


## Сравнение локального и reference SAE

Сравнение проводится на одинаковых свежих батчах MLP-активаций. `mse_per_activation` оценивает точность реконструкции отдельных координат, `mean_l0` — среднее число активных признаков на токен, а `reconstruction_score` показывает, насколько реконструкция сохраняет loss исходной language model.

Reference SAE ожидаемо должен быть сильнее короткого локального запуска: он обучался дольше и на существенно большем числе токенов.


In [14]:
@torch.no_grad()
def sae_activation_metrics(local_encoder, num_batches=5, batch_size=2048):
    rows = []
    for _ in tqdm.trange(num_batches):
        mlp_acts = sample_mlp_post_acts(batch_size)
        _, reconstruction, feature_acts, l2_loss, l1_loss = local_encoder(mlp_acts)
        active_mask = feature_acts > 0

        rows.append(
            {
                "mse_per_activation": (
                    reconstruction.float() - mlp_acts.float()
                ).pow(2).mean().item(),
                "l2_loss": l2_loss.item(),
                "l1_loss": l1_loss.item(),
                "mean_l0": active_mask.float().sum(dim=-1).mean().item(),
                "density": active_mask.float().mean().item(),
                "dead_feature_fraction": (
                    active_mask.float().sum(dim=0) == 0
                ).float().mean().item(),
            }
        )
    return pd.DataFrame(rows).mean().to_dict()


def compare_saes(reference_encoder, candidate_encoder=None):
    encoders = [
        ("NeelNanda/sparse_autoencoder", reference_encoder),
        ("local_trained_sae", candidate_encoder),
    ]
    rows = []

    for name, local_encoder in encoders:
        if local_encoder is None:
            continue

        row = {"sae": name, "d_hidden": local_encoder.d_hidden}
        row.update(sae_activation_metrics(local_encoder))
        score, model_loss, reconstruction_loss, zero_ablation_loss = get_recons_loss(
            num_batches=3,
            local_encoder=local_encoder,
        )
        row.update(
            {
                "model_loss": model_loss,
                "reconstruction_loss": reconstruction_loss,
                "zero_ablation_loss": zero_ablation_loss,
                "reconstruction_score": score,
            }
        )
        rows.append(row)

    return pd.DataFrame(rows)


sae_comparison_df = compare_saes(encoder, globals().get("trained_encoder"))
sae_comparison_df.style.format(
    {
        "mse_per_activation": "{:.5f}",
        "l2_loss": "{:.3f}",
        "l1_loss": "{:.3f}",
        "mean_l0": "{:.2f}",
        "density": "{:.5f}",
        "dead_feature_fraction": "{:.2%}",
        "model_loss": "{:.3f}",
        "reconstruction_loss": "{:.3f}",
        "zero_ablation_loss": "{:.3f}",
        "reconstruction_score": "{:.2%}",
    }
)


  0%|          | 0/5 [00:00<?, ?it/s]

loss: 3.2059, reconstruction: 3.7018, zero ablation: 8.7397
Reconstruction score: 91.04%


  0%|          | 0/5 [00:00<?, ?it/s]

loss: 3.2239, reconstruction: 4.6012, zero ablation: 8.7511
Reconstruction score: 75.08%


,sae,d_hidden,mse_per_activation,l2_loss,l1_loss,mean_l0,density,dead_feature_fraction,model_loss,reconstruction_loss,zero_ablation_loss,reconstruction_score
0,NeelNanda/sparse_autoencoder,16384,0.01568,32.120,11.174,139.02,0.00849,62.16%,3.206,3.702,8.740,91.04%
1,local_trained_sae,16384,0.03517,72.022,23.814,184.82,0.01128,76.31%,3.224,4.601,8.751,75.08%


## Проверка реконструкции

Сравним loss исходной модели с loss после подстановки реконструированных активаций и после полного зануления MLP. Эта проверка дополняет обычный MSE: небольшая ошибка в пространстве активаций не обязательно означает, что сохранено поведение модели.


In [15]:
_ = get_recons_loss(num_batches=5, local_encoder=encoder)


loss: 3.2484, reconstruction: 3.7341, zero ablation: 8.7699
Reconstruction score: 91.20%


## Анализ редких признаков


Сначала оценим частоту срабатывания каждого признака и построим распределение в логарифмической шкале. К частотам добавляется маленькая константа, чтобы признаки без единого срабатывания тоже попали на график.

Затем сравним encoder-направления редких признаков со средним редким направлением. Если большинство из них почти коллинеарны, это больше похоже на общий артефакт обучения, чем на набор разных содержательных признаков.


In [16]:
frequencies = get_feature_frequencies(num_batches=50, local_encoder=encoder)


  0%|          | 0/50 [00:00<?, ?it/s]

Доля признаков без активаций в выборке: 0.00%


In [17]:
# Добавка 10^-6.5 помещает признаки без срабатываний в левый край histogram.
log_frequencies = (frequencies + 10**-6.5).log10()
frequency_figure = px.histogram(
    x=utils.to_numpy(log_frequencies),
    histnorm="percent",
    title="Распределение частот SAE-признаков",
    labels={"x": "log10(частота)", "y": "Доля признаков, %"},
    template=PLOTLY_TEMPLATE,
)
frequency_figure.show()


In [18]:
is_rare = frequencies < 1e-4
rare_encoder_directions = encoder.W_enc[:, is_rare]
rare_mean_direction = rare_encoder_directions.mean(dim=-1)

cosine_similarity = (
    rare_mean_direction @ encoder.W_enc
    / rare_mean_direction.norm().clamp_min(1e-8)
    / encoder.W_enc.norm(dim=0).clamp_min(1e-8)
)
similarity_df = pd.DataFrame(
    {
        "cosine_similarity": utils.to_numpy(cosine_similarity),
        "rare": utils.to_numpy(is_rare),
    }
)
similarity_figure = px.histogram(
    similarity_df,
    x="cosine_similarity",
    color="rare",
    marginal="box",
    histnorm="percent",
    barmode="overlay",
    title="Сходство со средним направлением редких признаков",
    labels={
        "cosine_similarity": "Cosine similarity",
        "rare": "Редкий признак",
        "percent": "Доля признаков, %",
    },
    template=PLOTLY_TEMPLATE,
)
similarity_figure.show()


## Интерпретация отдельного признака


Разберём признак 7. Сначала найдём позиции с максимальной активацией в батче реальных данных, затем проверим его на коротком вручную заданном тексте и посмотрим на прямой вклад decoder-направления в logits.

Один список top activations не доказывает интерпретацию: контексты могут быть случайно похожи. Проверка на контролируемом тексте помогает увидеть, реагирует ли признак на предполагаемый паттерн.


In [19]:
feature_id = 7
batch_size = 128

print(f"Частота признака {feature_id}: {frequencies[feature_id].item():.4f}")


Частота признака 7: 0.0030


In [20]:
tokens = all_tokens[:batch_size]
hook_name = utils.get_act_name("post", 0)
_, cache = model.run_with_cache(
    tokens,
    stop_at_layer=1,
    names_filter=hook_name,
)
mlp_acts = cache[hook_name]
flat_mlp_acts = mlp_acts.reshape(-1, model.cfg.d_mlp)
hidden_acts = encoder.encode(flat_mlp_acts)

print(f"Форма MLP-активаций: {tuple(mlp_acts.shape)}")
print(f"Форма SAE-активаций после flatten: {tuple(hidden_acts.shape)}")


Форма MLP-активаций: (128, 128, 2048)
Форма SAE-активаций после flatten: (16384, 16384)


В колонке `context` текущий токен расположен между вертикальными чертами. Например, `·socialist·and|·he|·favors` означает, что SAE-активация относится к токену `·he`, а соседние токены показаны только для контекста.

Сортировка по `activation` позволяет быстро сформулировать первую гипотезу о значении признака.


In [21]:
token_df = make_token_df(tokens)
token_df["activation"] = utils.to_numpy(hidden_acts[:, feature_id])
token_df.sort_values("activation", ascending=False).head(20).round(
    {"activation": 4}
)


,str_tokens,unique_token,context,batch,pos,label,activation
7989,·we,·we/53,ors·and·even·heroes·and|·we|·feel,62,53,62/53,1.7227
14648,·he,·he/56,·1971·and|·he|·registered,114,56,114/56,1.2116
6643,·we,·we/115,·anchor·[xx]·but|·we|·want,51,115,51/115,0.7982
1546,·I,·I/10,ing·experience–and·while|·I|·continue,12,10,12/10,0.5508
7618,·you,·you/66,"·family·law·issues,·then|·you|·came",59,66,59/66,0.4081
16361,·there,·there/105,"·simple,·honest·mistakes·and|·there|·""",127,105,127/105,0.3887
4630,·y,·y/22,"·Super·friendly,·affordable·and|·y|ummy",36,22,36/22,0.3591
1559,I,I/23,·amazing·platform·for·creators–|I|’,12,23,12/23,0.2856
7961,·people,·people/25,·rich·cultural·heritage·and·the|·people|·who,62,25,62/25,0.2741
13948,·there,·there/124,"·documentation·for·these,·but|·there|·are",108,124,108/124,0.2724


### Проверка на заданном тексте

Таблица сохраняет точные значения, а Plotly-график показывает распределение активаций вдоль последовательности. Текст можно менять прямо в следующей ячейке и повторно запускать обе визуализации.


In [22]:
starting_text = (
    "Hero and I will head to Samantha and Mark's, then he and she will. "
    "Then I or you"
)

feature_activation_table(starting_text, feature_id).round(
    {"activation": 4}
)


,position,token,activation
0,0,<|BOS|>,0.0000
1,1,H,0.0000
2,2,ero,0.0000
3,3,·and,0.0000
4,4,·I,1.6312
5,5,·will,0.2436
6,6,·head,0.0000
7,7,·to,0.0000
8,8,·Sam,0.0000
9,9,antha,0.0000


In [23]:
plot_feature_activations(starting_text, feature_id)


### Влияние признака на logits

В однослойной модели decoder-направление SAE попадает в residual stream через `W_out`, а затем преобразуется в logits матрицей `W_U`. Произведение ниже показывает прямой logit effect выбранного признака: положительные значения соответствуют токенам, вероятность которых признак склонен повышать.

Это локальная линейная оценка. Она не учитывает нормализацию и взаимодействие с остальными компонентами residual stream, но хорошо подходит для первичной интерпретации.


In [24]:
logit_effect = encoder.W_dec[feature_id] @ model.W_out[0] @ model.W_U
create_vocab_df(logit_effect).head(20).round(
    {"logit": 4}
)


,token,logit
1782,'ll,1.8750
6835,eding,1.5589
5440,·certainly,1.5222
3409,·hope,1.5211
29310,OULD,1.5086
4931,·wouldn,1.4833
44469,cheon,1.4356
7754,·definitely,1.3975
1606,·seem,1.3944
41304,·sincerely,1.3640
